### Step 1: Import libraries and API Keys

In [22]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key not found")

### Step 2: Setup Pushover

In [23]:
pushover_user= os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print(f"PUSHOVER_USER: {pushover_user}")
print(f"PUSHOVER_TOKEN: {pushover_token}")

PUSHOVER_USER: usirayudtxm7k5nn6jj39qczvaqibj
PUSHOVER_TOKEN: a7macpjd2i18up2bmekhwf53p94ewm


In [24]:
# Test Pushover API

import requests
def send_notification(message: str):
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message
    }
    requests.post(pushover_url, data=payload)

In [25]:
send_notification("Hello GK, from AI engineering course. A test message from the API")

### Step 3: Describe Pushover as as LLM tool

In [26]:
send_notification_function = {
    "name": "send_notification",
    "description": "Send a push notifcation to the user phone using pushover. Use this to alert the user about important events or time sensitive information.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notifiication message to send to the user device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for the LLM

In [27]:
tools = [{
    "type": "function",
    "function": send_notification_function
}]

### Step 5: Calling the tool from an LLM

In [28]:
client = OpenAI()
response = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Please send me a notification telling me what amazing progress i am making in the AI engineering course."}
    ],
    tools=tools
)

# Check if model wants to call a tool
message = response.choices[0].message

In [29]:
# print(message)
print(json.dumps(message.model_dump(), indent=2))

{
  "content": null,
  "refusal": null,
  "role": "assistant",
  "annotations": [],
  "audio": null,
  "function_call": null,
  "tool_calls": [
    {
      "id": "call_OEC0jx1i00ma8FfqxCRTzWhC",
      "function": {
        "arguments": "{\"message\":\"You're making amazing progress in your AI engineering course! Keep up the great work!\"}",
        "name": "send_notification"
      },
      "type": "function"
    }
  ]
}


In [30]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)

# Actually send the notification using the tool
    send_notification(args["message"])
    print(f" Sent notification: {args['message']}")
else:
    print(message.content)


 Sent notification: You're making amazing progress in your AI engineering course! Keep up the great work!
